In [2]:
import sys
sys.path.append("..")

from src.spark_utils import load_config, get_spark, read_raw_csv, prepare_table, write_to_mysql
from src.table_configs import TABLES, LOAD_ORDER, rename_map, type_map

In [3]:
cfg = load_config()
spark = get_spark(cfg["jdbc_jar"])

26/09/05 15:00:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [4]:
for table_key in LOAD_ORDER:
    table_cfg = TABLES[table_key]
    file_path = cfg["raw_dir"] / table_cfg["file"]

    print(f"Loading: {table_key} <- {table_cfg['file']}")

    raw_df = read_raw_csv(spark, file_path)

    clean_df = prepare_table(
        raw_df,
        col_rename=rename_map(table_key),
        col_types=type_map(table_key),
    )

    write_to_mysql(
        clean_df,
        table_cfg["db_table"],
        cfg["jdbc_url"],
        cfg["db_user"],
        cfg["db_password"],
    )

    print(f"Done: {table_key}")

Loading: district <- district.csv


26/09/05 15:10:17 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Done: district
Loading: client <- client.asc
Done: client
Loading: account <- account.asc
Done: account
Loading: disp <- disp.asc
Done: disp
Loading: card <- card.asc
Done: card
Loading: loan <- loan.asc
Done: loan
Loading: order <- order.asc
Done: order
Loading: trans <- trans.asc


Done: trans


In [5]:
spark.stop()